# Étape 2 — Feature Engineering

Objectif : créer de nouvelles variables à partir des données brutes pour enrichir les modèles.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

df = pd.read_parquet('../data/yellow_tripdata_clean.parquet')
print(f'Shape : {df.shape}')

## 2.1 Variables temporelles obligatoires

In [ ]:
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

df['duree_course'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60
df = df[df['duree_course'] > 0]

df['heure_journee'] = df['tpep_pickup_datetime'].dt.hour
df['jour_semaine']  = df['tpep_pickup_datetime'].dt.dayofweek
df['est_weekend']   = (df['jour_semaine'] >= 5).astype(int)
df['est_heure_pointe'] = df['heure_journee'].isin([7,8,9,17,18,19]).astype(int)

df[['duree_course','heure_journee','jour_semaine','est_weekend','est_heure_pointe']].head()

## 2.2 Variables bonus

In [ ]:
df['vitesse_moyenne'] = df['trip_distance'] / (df['duree_course'] / 60)
df['vitesse_moyenne'] = df['vitesse_moyenne'].replace([np.inf, -np.inf], np.nan).fillna(0)

df['est_trajet_aeroport'] = df['RateCodeID'].isin([2, 3]).astype(int)

df['heure_sin'] = np.sin(2 * np.pi * df['heure_journee'] / 24)
df['heure_cos'] = np.cos(2 * np.pi * df['heure_journee'] / 24)

print('Nouvelles colonnes :', ['vitesse_moyenne','est_trajet_aeroport','heure_sin','heure_cos'])

## 2.3 Visualisation des nouvelles variables

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.groupby('heure_journee').size().plot(ax=axes[0], marker='o')
axes[0].set_title('Nombre de courses par heure')
axes[0].set_xlabel('Heure')

df['duree_course'].clip(0, 60).hist(bins=50, ax=axes[1], color='coral')
axes[1].set_title('Distribution durée (min)')

plt.tight_layout()
plt.savefig('../figures/02_features.png', dpi=150)
plt.show()

In [ ]:
df.to_parquet('../data/yellow_tripdata_features.parquet', index=False)
print('Données avec features sauvegardées.')